In [1]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

df = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# NaN 50% 넘는 열 자동 삭제
df = df.drop(columns=df.columns[df.isnull().mean() > 0.5])
test = test.drop(columns=test.columns[test.isnull().mean() > 0.5])

# 공통 열만 맞추기
common_cols = df.columns.intersection(test.columns)
df = df[list(common_cols) + ["SalePrice"]]
test = test[common_cols]

# 숫자 열 NaN 중간값으로 채우기
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
test[test.select_dtypes(include="number").columns] = test[test.select_dtypes(include="number").columns].fillna(test[test.select_dtypes(include="number").columns].median())

# 문자열 열 숫자로 변환
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].astype("category").apply(lambda x: x.cat.codes)
test[str_cols] = test[str_cols].astype("category").apply(lambda x: x.cat.codes)

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]
X_test = test

# 교차검증
model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
scores = cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
print("교차검증 MAE:", -scores.mean())

# 학습 및 예측
model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": predictions
})
submission.to_csv("submission.csv", index=False)
print("저장 완료!")

C:\Users\kimgy\AppData\Local\Temp\ipykernel_1424\1231002426.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns


교차검증 MAE: 17838.732009616564
저장 완료!
